In [25]:
import pandas as pd
import seaborn as sns
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [26]:
df = pd.read_csv("datas/08-seismic_activity_svm.csv")
df.head()

,underground_wave_energy,vibration_axis_variation,seismic_event_detected
0,9.539392,-3.000000,0
1,9.558241,-2.939394,0
2,9.576669,-2.878788,0
3,9.594678,-2.818182,0
4,9.612272,-2.757576,0


In [27]:
X = df[["underground_wave_energy","vibration_axis_variation"]].values
y = df["seismic_event_detected"].values

In [ ]:
sns.scatterplot(x=df["underground_wave_energy"], y=df["vibration_axis_variation"], hue=df["seismic_event_detected"])
plt.show()

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [18]:
len(X_train), len(X_test), len(y_train), len(y_test)

(320, 80, 320, 80)

In [19]:
print(f"Xtrain shape: {X_train.shape}")
print(f"Xtest shape: {X_test.shape}")
print(f"ytrain shape: {y_train.shape}")
print(f"ytest shape: {y_test.shape}")

Xtrain shape: (320, 2)
Xtest shape: (80, 2)
ytrain shape: (320,)
ytest shape: (80,)


In [20]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [21]:
def calculate_accuracy(y_test, y_pred):
    correct = torch.eq(y_test, y_pred).sum().item()
    accuracy = (correct / len(y_pred)) * 100
    return accuracy

In [22]:
from torch import nn
class NonLinearClassModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.non_linear_layer1 = nn.Linear(in_features=2, out_features=5)
        self.non_linear_layer2 = nn.Linear(in_features=5, out_features=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.non_linear_layer2(self.relu(self.non_linear_layer1(x)))

In [23]:
model_1 = NonLinearClassModel()

loss_fn = nn.BCEWithLogitsLoss()

optimizer_2 = torch.optim.Adam(params=model_1.parameters(), lr=0.01)

In [24]:
torch.manual_seed(42)
epochs = 120

for epoch in range(epochs):
    model_1.train()

    y_logits = model_1(X_train)
    y_preds = torch.round(torch.sigmoid(y_logits))

    loss = loss_fn(y_logits, y_train)
    acc = calculate_accuracy(y_train, y_preds)

    optimizer_2.zero_grad()
    loss.backward()
    optimizer_2.step()

    model_1.eval()

    with torch.inference_mode():
        test_logits = model_1(X_test)
        test_pred = torch.round(torch.sigmoid(test_logits))

        loss_test = loss_fn(test_logits, y_test)
        acc_test = calculate_accuracy(y_test, test_pred)

    if epoch % 5 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {loss_test:.5f}, Test acc: {acc_test:.2f}%")

Epoch: 0 | Loss: 0.69917, Accuracy: 48.75% | Test loss: 0.72457, Test acc: 42.50%
Epoch: 5 | Loss: 0.68331, Accuracy: 51.88% | Test loss: 0.69799, Test acc: 45.00%
Epoch: 10 | Loss: 0.67332, Accuracy: 53.75% | Test loss: 0.68025, Test acc: 51.25%
Epoch: 15 | Loss: 0.66661, Accuracy: 54.37% | Test loss: 0.66788, Test acc: 55.00%
Epoch: 20 | Loss: 0.66181, Accuracy: 53.44% | Test loss: 0.65877, Test acc: 55.00%
Epoch: 25 | Loss: 0.65820, Accuracy: 52.81% | Test loss: 0.65180, Test acc: 55.00%
Epoch: 30 | Loss: 0.65524, Accuracy: 52.19% | Test loss: 0.64629, Test acc: 55.00%
Epoch: 35 | Loss: 0.65273, Accuracy: 51.56% | Test loss: 0.64178, Test acc: 55.00%
Epoch: 40 | Loss: 0.65048, Accuracy: 51.25% | Test loss: 0.63800, Test acc: 55.00%
Epoch: 45 | Loss: 0.64838, Accuracy: 51.25% | Test loss: 0.63478, Test acc: 55.00%
Epoch: 50 | Loss: 0.64642, Accuracy: 50.94% | Test loss: 0.63196, Test acc: 55.00%
Epoch: 55 | Loss: 0.64453, Accuracy: 50.94% | Test loss: 0.62945, Test acc: 55.00%
Epoch: